In [1]:
# need to read in the BATS cast sheets and pull the c/c/n information I need...I have written this for MATLAB but not Python

In [3]:
import os
import re
import pandas as pd
import xlwings as xw #need to use this to make the new sheet to keep the Dalhousie validations
import pdb #use with set_trace() 

In [5]:
data_dir = 'seq_data'

In [7]:
os.listdir(data_dir)

['10412_biosscope.xlsx',
 '10414_biosscope.xlsx',
 '10415_biosscope.xlsx',
 '10416_biosscope.xlsx',
 '10417_biosscope.xlsx',
 '10418_biosscope.xlsx',
 '10419_biosscope.xlsx',
 '10420_biosscope.xlsx',
 '10421_biosscope.xlsx',
 '10422_biosscope.xlsx',
 '10423_biosscope.xlsx',
 '10424_biosscope.xlsx',
 '10425_biosscope.xlsx',
 '10426_biosscope.xlsx',
 '10427_biosscope.xlsx',
 '10428_biosscope.xlsx',
 '10429_biosscope.xlsx',
 '10430_biosscope.xlsx',
 '20413_biosscope.xlsx',
 '20414_biosscope.xlsx',
 '20415_biosscope.xlsx',
 '20423_biosscope.xlsx',
 'BIOS-SCOPE Archive DNA Submission July 2022.xlsx',
 'biosscope_combinedCCN.xlsx',
 'IMR-SubmissionTemplate_V4_4_BIOSSCOPE 2026 Sept.xlsx',
 'Sample_sheets_20260915']

In [8]:
df = pd.read_excel(os.path.join(data_dir,'BIOS-SCOPE Archive DNA Submission July 2022.xlsx'),sheet_name = 'Submission')
df.head()

,Project,Institute,Date,Gene,Target,Primers,Seq_Run,Lane,Plate,Well,Tube Name,Sample ID,Cruise,Depth,P1 Sequence,Illumina Primer 2,P2 Sequence,Primer ID
0,Archive DNA,OSU,2022-07-02,16s,v4,515F; 806RB,Run #993,5,4,A11,Giov 5 V4,168_0,168,0,CTACTATA,SA711,GTCTATGA,SB501_SA711
1,Archive DNA,OSU,2022-07-02,16s,v4,515F; 806RB,Run #993,5,4,C11,Giov 5 V4,169_0,169,0,AGAGTCAC,SA711,GTCTATGA,SB503_SA711
2,Archive DNA,OSU,2022-07-02,16s,v4,515F; 806RB,Run #993,5,4,B11,Giov 5 V4,169_200,169,200,CGTTACTA,SA711,GTCTATGA,SB502_SA711
3,Archive DNA,OSU,2022-07-02,16s,v4,515F; 806RB,Run #993,5,4,E11,Giov 5 V4,170_0,170,0,ACGTCTCG,SA711,GTCTATGA,SB505_SA711
4,Archive DNA,OSU,2022-07-02,16s,v4,515F; 806RB,Run #993,5,4,G11,Giov 5 V4,171_0,171,0,GATCGTGT,SA711,GTCTATGA,SB507_SA711


In [17]:
c = df['Sample ID'].value_counts().reset_index()
c.columns = ['Value', 'Count']
dfd = c[c['Count'] > 2]
print(dfd)

                 Value  Count
0               ref330      6
1                ref20      6
2            mock even      6
3         mock stagger      6
4                  neg      6
5           AE1703_160      5
6   Positive_Mock_Even      4
7           AE1703_300      4
8           AE1703_250      4
9           AE1703_200      4
10        MOCK_EVEN_BC      4
11        MOCK_STAG_BC      4
12             NEG1_BC      4
13             NEG2_BC      4
14         Neg_Control      4
15         REF_330M_BC      4
16          REF_20M_BC      4
17  Positive_Mock_Stag      3
18             132-200      3
19          AE1703_120      3
20           AE1703_80      3
21    Positive_Ref_330      3


In [16]:
dfd.to_excel('seq_data/dups.xlsx')

In [11]:
c.head()

Sample ID
ref330          6
ref20           6
mock even       6
mock stagger    6
neg             6
Name: count, dtype: int64

In [70]:
for fs in folders:
    files = os.listdir(os.path.join(data_dir,fs))
    
    for fi in files:
        df = pd.read_excel(os.path.join(data_dir,fs,fi),nrows=0)
        fc = df.columns.tolist()
        fc = fc[0].lower()
        if 'bios-scope' in fc:
            #now read in the whole file
            df = pd.read_excel(os.path.join(data_dir,fs,fi),skiprows = 1)
            #delete the row with the BATS sample numbers:
            df = df.drop(df.index[0]).reset_index(drop = True)

            #look for BS_eDNA ...and pull sample numbers
            df = df[pd.notna(df['BS_eDNA'])].reset_index(drop = True)

            #make the NewID, will have to pad Niskin
            df['NewID'] = df['% cast'].astype(str) + df['Niskin'].astype(str).str.zfill(2)

            #I think the easy export is to write a new Excel file with what I need...may change my mind later
            df = df[['NewID','BS_eDNA','Depth','Flag']]
            #drop the last row that repeats the top row
            df = df.drop(df.index[-1]) 
            df.to_excel(os.path.join('seq_data',fi[:5] + '_biosscope.xlsx'))

In [32]:
fs = folders[2]

In [68]:
for fi in files:
    df = pd.read_excel(os.path.join(data_dir,fs,fi),nrows=0)
    fc = df.columns.tolist()
    fc = fc[0].lower()
    if 'bios-scope' in fc:
        #now read in the whole file
        df = pd.read_excel(os.path.join(data_dir,fs,fi),skiprows = 1)
        #delete the row with the BATS sample numbers:
        df = df.drop(df.index[0]).reset_index(drop = True)

        #look for BS_eDNA ...and pull sample numbers
        df = df[pd.notna(df['BS_eDNA'])].reset_index(drop = True)

        #make the NewID, will have to pad Niskin
        df['NewID'] = df['% cast'].astype(str) + df['Niskin'].astype(str).str.zfill(2)
        
        #I think the easy export is to write a new Excel file with what I need...may change my mind later
        df = df[['NewID','BS_eDNA','Depth','Flag']]
        #drop the last row that repeats the top row
        df = df.drop(df.index[-1]) 
        df.to_excel(os.path.join('seq_data',fi[:5] + '_biosscope.xlsx'))
        
    

> c:\users\laneylab\appdata\local\temp\ipykernel_6488\3258838590.py(20)<module>()



ipdb>  df.head()


     % cast Niskin Depth Flag  temp BS_HDOM BS_HDOM.1 BS_eDNA AW_cocc  \
0  10414012      1     1    1    20       1         1       1     NaN   
1  10414012      3    40    1    20       2         2       2     NaN   
2  10414012      5    80    1  19.7       3         3       3     NaN   
3  10414012      8   120    1  18.4       4         4       4     NaN   
4  10414012     10   160    1  19.3       5         5       5     NaN   

  LB_fungi LB_fungi.1 TR_HPLC TR_whole Niskin.1 Unnamed: 14       NewID  
0      NaN        NaN     NaN      NaN        1         NaN  1041401201  
1      NaN        NaN     2.4      2.1        3         NaN  1041401203  
2      NaN        NaN     3.4      3.2        5         NaN  1041401205  
3      NaN        NaN     5.4      5.2        8         NaN  1041401208  
4      NaN        NaN     7.4      NaN       10         NaN  1041401210  


ipdb>  df = df[['NewID','BS_eDNA','Depth'],]


*** pandas.errors.InvalidIndexError: (['NewID', 'BS_eDNA', 'Depth'],)


ipdb>          df = df[['NewID','BS_eDNA','Depth']]
ipdb>  df.head()


... (7 outputs left out)

ipdb>  fi[:5] + 'biosscope.xlsx'


'10414biosscope.xlsx'


ipdb>          df.to_excel(os.path.join('seq_data',fi[:5] + '_biosscope.xlsx'))
ipdb>  q


BdbQuit: 